# Photos Repair Semi-Automated Asset Copy / Import Workflow

Purpose:

This notebook exports selected **missing assets** from `backup_20250317` into a normal Finder folder, while preserving the original filenames.  
It also writes a metadata table for manual repair after importing the exported files into the current iCloud Photos Library.

This workflow is **copy-only**:

- It does not delete anything.
- It does not modify the backup Photos Library.
- It does not modify the current iCloud Photos Library.
- It only copies original files out of the backup library package into an export folder.

Human workflow after export:

1. Import the exported files into the current iCloud Photos Library.
2. Use the generated metadata TSV to manually restore caption, keywords, favorite, and album/folder membership where needed.
3. Treat album/folder restoration as a separate manual or later-scripted repair step.


## Workflow logic

Lookup after import should be based mainly on:

- `export_filename`
- `original_filename`
- `date`
- visible image/video content

The script intentionally keeps:

```text
export_filename == original_filename
```

No sequence prefix is added, because preserving the original filename is more important.

Important limitation:

Photos import does **not** recreate Photos album/folder structure from Finder folders.  
The export folder is only a transport container, not the final Photos organization.

The generated metadata table records album/folder paths so that they can be repaired manually or by a later dedicated metadata-repair script.


In [4]:
from pathlib import Path
import gzip
import pickle
import shutil
import csv
import json
from datetime import datetime

print("Notebook ready.")

Notebook ready.


## Configuration

Edit these paths if your notebook location or inventory cache folder is different.

Expected input:

- `backup_20250317.inventory.pkl.gz`
- the missing candidate UUID list from Test 2

Output:

- a Finder folder on Desktop containing exported media files
- `photos_repair_metadata.tsv`
- `photos_repair_metadata.json`
- `photos_repair_export_log.txt`


In [5]:
# ============================================================
# Configuration
# ============================================================

PROJECT_ROOT = Path.cwd()

BACKUP_INVENTORY_CACHE_PATH = PROJECT_ROOT / "data" / "inventory_cache" / "backup_20250317.inventory.pkl.gz"

EXPORT_DIR = Path.home() / "Desktop" / "Photos_Repair_Export_20260605"

METADATA_TSV_PATH = EXPORT_DIR / "photos_repair_metadata.tsv"
METADATA_JSON_PATH = EXPORT_DIR / "photos_repair_metadata.json"
EXPORT_LOG_PATH = EXPORT_DIR / "photos_repair_export_log.txt"

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("BACKUP_INVENTORY_CACHE_PATH:", BACKUP_INVENTORY_CACHE_PATH)
print("EXPORT_DIR:", EXPORT_DIR)

PROJECT_ROOT: /Users/huohsien/workspace/python/explore_photos_library
BACKUP_INVENTORY_CACHE_PATH: /Users/huohsien/workspace/python/explore_photos_library/data/inventory_cache/backup_20250317.inventory.pkl.gz
EXPORT_DIR: /Users/huohsien/Desktop/Photos_Repair_Export_20260605


## Load backup inventory

This reads the already-built inventory cache.  
It does not rescan the Photos Library.


In [6]:
def load_inventory_pickle_gz(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Inventory cache does not exist: {path}")

    with gzip.open(path, "rb") as f:
        return pickle.load(f)

inventory_backup = load_inventory_pickle_gz(BACKUP_INVENTORY_CACHE_PATH)

backup_assets = inventory_backup.get("assets", [])
backup_asset_by_uuid = {
    asset.get("uuid"): asset
    for asset in backup_assets
    if asset.get("uuid")
}

print("Loaded backup inventory.")
print("backup asset count:", len(backup_assets))
print("backup asset UUID index count:", len(backup_asset_by_uuid))

FileNotFoundError: Inventory cache does not exist: /Users/huohsien/workspace/python/explore_photos_library/data/inventory_cache/backup_20250317.inventory.pkl.gz

## Missing candidates to export

Current manual state:

- `IMG_0003.PNG` was not exported because current iCloud Photos already has a corresponding JPG at the same timestamp; caption was manually copied.
- `IMG_0024.JPG` was already copied successfully using PowerPhotos.
- The remaining 15 candidates are exported here.

If you later decide to include/exclude other assets, edit this list.


In [ ]:
# ============================================================
# Remaining ASSET_MISSING_FROM_CURRENT candidates
# ============================================================

repair_candidate_backup_uuids = [
    "182137B0-D3F4-46FC-9499-0BB32E977ED9",  # 03 IMG_0106.JPG
    "DFC351E0-5BE6-4FBD-97EC-09BEE5141535",  # 04 IMG_0222.JPG
    "BC5E512B-BBB8-4FAB-8105-25B277007751",  # 05 IMG_0263.JPG
    "98001795-CF8F-479E-A555-75C570F87FBB",  # 06 IMG_0537.JPG
    "EE1D4DA4-20A7-418B-AD25-4F3AECA0F5A9",  # 07 IMG_0652.JPG
    "99B5E7AC-2727-489F-9E42-CBBA69A4385F",  # 08 IMG_0912.JPG
    "6190676F-FBA4-41D3-B99A-E1B5CE1035ED",  # 09 IMG_1425.JPG
    "5D7264FA-5815-40CF-BD96-BE8A4EADD618",  # 10 IMG_1596.JPG
    # "6B815B80-CDA8-4AC0-8DD6-1C7EFF9BF84A",  # 11 IMG_1894.PNG
    # "5AD071AD-911C-425C-AF8F-542A8FD02D36",  # 12 IMG_1918.JPG
    "E2A18EB3-D9AE-40EA-94E7-E8B2B0B47C62",  # 13 IMG_3669.JPG
    "EEC70EEA-4C79-4F02-886B-FD51AEEE32B3",  # 14 IMG_3689.JPG
    "AA0BC72D-CAA7-4DE6-9C12-7FCEC3836395",  # 15 IMG_4623.JPG
    "20F498F3-F116-4B7F-AF01-FDED12D1DBAA",  # 16 IMG_6311.JPG
    "83052E5C-2718-4FE0-B143-7C8DD47F8568",  # 17 IMG_8638.jpeg
]

print("candidate count:", len(repair_candidate_backup_uuids))

## Metadata helpers

`album_paths` is the important field for later manual repair.

Because Photos can have folders containing albums, and the same album title can appear in different folder contexts, the useful repair record should be a path-like representation, not just a bare album title.

This helper is defensive because inventory structures can evolve.


In [ ]:
def _stringify_value(value):
    if value is None:
        return ""
    if isinstance(value, (str, int, float, bool)):
        return str(value)
    return json.dumps(value, ensure_ascii=False, sort_keys=True)


def asset_album_path_strings(asset):
    """
    Return full album paths for manual album/folder repair.

    Output examples:

    Root-level album:
        Pinterest Girls

    Album inside folder:
        Girls / Pinterest Girls

    Album inside nested folders:
        Imported / 2018 / Pinterest Girls

    This uses the inventory structure created by photos_inventory.py:
    asset["albums"] is album_uuid -> album object
    album["folders"] is folder_uuid -> folder object
    folder["path"] is the full folder path up to that folder
    """

    albums = asset.get("albums") or {}

    if not albums:
        return []

    if isinstance(albums, dict):
        album_records = list(albums.values())
    elif isinstance(albums, list):
        album_records = albums
    else:
        return [_stringify_value(albums)]

    album_paths = []

    for album in album_records:
        if isinstance(album, str):
            album_paths.append(album)
            continue

        if not isinstance(album, dict):
            album_paths.append(_stringify_value(album))
            continue

        album_title = album.get("title") or album.get("name") or album.get("album_title")

        if not album_title:
            album_paths.append(_stringify_value(album))
            continue

        folders = album.get("folders") or {}

        if isinstance(folders, dict):
            folder_records = list(folders.values())
        elif isinstance(folders, list):
            folder_records = folders
        else:
            folder_records = []

        folder_paths = []

        for folder in folder_records:
            if isinstance(folder, str):
                folder_paths.append(folder)
            elif isinstance(folder, dict):
                folder_path = folder.get("path")
                if folder_path:
                    folder_paths.append(folder_path)

        if folder_paths:
            # folder["path"] already contains the full path up to that folder.
            # If multiple folder objects exist for the same album, choose the deepest path.
            deepest_folder_path = max(
                folder_paths,
                key=lambda p: (p.count("/"), len(p)),
            )
            album_paths.append(f"{deepest_folder_path} / {album_title}")
        else:
            # Root-level album.
            album_paths.append(album_title)

    return sorted(set(path for path in album_paths if path))

def asset_keyword_string(asset):
    keywords = asset.get("keywords") or []
    if isinstance(keywords, str):
        return keywords
    return " | ".join(_stringify_value(x) for x in keywords if x)


def asset_caption(asset):
    return (
        asset.get("description")
        or asset.get("caption")
        or asset.get("title")
        or ""
    )

## Dry run

This does not copy files.  
It checks:

- UUID exists in backup inventory
- source path exists
- export filename is the same as original filename
- no export filename collision

If a filename collision happens, the notebook stops.  
It does **not** auto-rename, because preserving filenames is part of the plan.


In [ ]:
def build_export_plan(candidate_uuids):
    rows = []
    errors = []
    export_names_seen = set()

    for backup_uuid in candidate_uuids:
        asset = backup_asset_by_uuid.get(backup_uuid)

        if asset is None:
            errors.append({
                "backup_uuid": backup_uuid,
                "error": "UUID_NOT_FOUND_IN_BACKUP_INVENTORY",
            })
            continue

        source_path_string = asset.get("path")
        original_filename = asset.get("original_filename") or asset.get("filename")

        if not original_filename:
            errors.append({
                "backup_uuid": backup_uuid,
                "error": "MISSING_ORIGINAL_FILENAME",
            })
            continue

        export_filename = original_filename

        if export_filename in export_names_seen:
            errors.append({
                "backup_uuid": backup_uuid,
                "original_filename": original_filename,
                "export_filename": export_filename,
                "error": "DUPLICATE_EXPORT_FILENAME_IN_THIS_BATCH",
            })
            continue

        export_names_seen.add(export_filename)

        if source_path_string is None:
            errors.append({
                "backup_uuid": backup_uuid,
                "original_filename": original_filename,
                "error": "SOURCE_PATH_NONE",
            })
            continue

        source_path = Path(source_path_string)

        if not source_path.exists():
            errors.append({
                "backup_uuid": backup_uuid,
                "original_filename": original_filename,
                "source_path": str(source_path),
                "error": "SOURCE_PATH_DOES_NOT_EXIST",
            })
            continue

        export_path = EXPORT_DIR / export_filename

        if export_path.exists():
            errors.append({
                "backup_uuid": backup_uuid,
                "original_filename": original_filename,
                "export_path": str(export_path),
                "error": "EXPORT_FILE_ALREADY_EXISTS",
            })
            continue

        row = {
            # Lookup / identity after import
            "export_filename": export_filename,
            "original_filename": original_filename,
            "date": _stringify_value(asset.get("date")),

            # Human metadata repair
            "description_caption": asset_caption(asset),
            "keywords": asset_keyword_string(asset),
            "favorite": _stringify_value(asset.get("favorite")),
            "hidden": _stringify_value(asset.get("hidden")),
            "album_paths": " || ".join(asset_album_path_strings(asset)),

            # Source/debug trace
            "backup_uuid": backup_uuid,
            "source_path": str(source_path),
            "export_path": str(export_path),

            # Secondary reference, not expected to be manually restorable
            "date_added_backup": _stringify_value(asset.get("date_added")),
            "is_movie": _stringify_value(asset.get("is_movie")),
        }

        rows.append(row)

    return rows, errors


export_plan_rows, export_plan_errors = build_export_plan(repair_candidate_backup_uuids)

print("=" * 80)
print("Dry run")
print("=" * 80)
print("planned exports:", len(export_plan_rows))
print("errors:", len(export_plan_errors))

if export_plan_errors:
    print()
    print("Errors")
    print("-" * 80)
    for error in export_plan_errors:
        print(error)

print()
print("Planned export rows")
print("-" * 80)
for row in export_plan_rows:
    print(
        row["export_filename"],
        "| date:", row["date"],
        "| caption:", row["description_caption"],
        "| favorite:", row["favorite"],
        "| albums:", row["album_paths"],
    )

## Execute copy-only export

Run this cell only after the dry run looks correct.

It copies files from the backup Photos Library package to the export folder while preserving original filenames.

It also writes:

- TSV for human repair
- JSON for complete trace/debug
- plain-text log


In [ ]:
if export_plan_errors:
    raise RuntimeError("Dry run has errors. Fix them before copying.")

fieldnames = [
    # Lookup / identity after import
    "export_filename",
    "original_filename",
    "date",

    # Human metadata repair
    "description_caption",
    "keywords",
    "favorite",
    "hidden",
    "album_paths",

    # Source/debug trace
    "backup_uuid",
    "source_path",
    "export_path",

    # Secondary reference
    "date_added_backup",
    "is_movie",
]

for row in export_plan_rows:
    shutil.copy2(row["source_path"], row["export_path"])

with METADATA_TSV_PATH.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter="\t")
    writer.writeheader()
    writer.writerows(export_plan_rows)

with METADATA_JSON_PATH.open("w", encoding="utf-8") as f:
    json.dump(
        {
            "created_at": datetime.now().isoformat(timespec="seconds"),
            "purpose": "Copy-only export of ASSET_MISSING_FROM_CURRENT candidates from backup_20250317 for later import into current iCloud Photos Library.",
            "export_dir": str(EXPORT_DIR),
            "metadata_tsv_path": str(METADATA_TSV_PATH),
            "metadata_json_path": str(METADATA_JSON_PATH),
            "exported_count": len(export_plan_rows),
            "errors": export_plan_errors,
            "rows": export_plan_rows,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

with EXPORT_LOG_PATH.open("w", encoding="utf-8") as f:
    f.write("Photos repair copy-only export\n")
    f.write("=" * 80 + "\n")
    f.write(f"created_at: {datetime.now().isoformat(timespec='seconds')}\n")
    f.write(f"export_dir: {EXPORT_DIR}\n")
    f.write(f"metadata_tsv_path: {METADATA_TSV_PATH}\n")
    f.write(f"metadata_json_path: {METADATA_JSON_PATH}\n")
    f.write(f"exported_count: {len(export_plan_rows)}\n")
    f.write("\n")
    for row in export_plan_rows:
        f.write(f"{row['export_filename']}\n")
        f.write(f"  date: {row['date']}\n")
        f.write(f"  caption: {row['description_caption']}\n")
        f.write(f"  keywords: {row['keywords']}\n")
        f.write(f"  favorite: {row['favorite']}\n")
        f.write(f"  hidden: {row['hidden']}\n")
        f.write(f"  album_paths: {row['album_paths']}\n")
        f.write(f"  source_path: {row['source_path']}\n")
        f.write(f"  export_path: {row['export_path']}\n")
        f.write("\n")

print("=" * 80)
print("Copy-only export finished")
print("=" * 80)
print("export dir:", EXPORT_DIR)
print("metadata TSV:", METADATA_TSV_PATH)
print("metadata JSON:", METADATA_JSON_PATH)
print("export log:", EXPORT_LOG_PATH)
print("exported files:", len(export_plan_rows))

## After export: import into current iCloud Photos Library

Manual Photos step:

1. Open the current iCloud Default Photos Library in Apple Photos.
2. Use `File → Import...`.
3. Select the exported image/video files inside:

```text
~/Desktop/Photos_Repair_Export_20260605/
```

4. Import them.
5. Keep the import result visible using `Imports` / `Last Import` if possible.
6. Use `photos_repair_metadata.tsv` to restore:
   - caption
   - keywords
   - favorite
   - album/folder membership

Do not expect the Finder export folder to recreate Photos albums/folders.


## Metadata repair priority

After import, repair in this order:

1. Caption / description  
   Usually the most visible human metadata.

2. Favorite  
   Easy to restore manually.

3. Keywords  
   Useful but more annoying to enter manually.

4. Album/folder membership  
   Hardest part. Use `album_paths` as the authoritative reference.
   This may need a separate dedicated workflow.

5. Hidden  
   Lower priority. Treat as transient unless the item clearly matters.
